In [ ]:
from pathlib import Path
from pypdf import PdfReader


def resolve_data_path():
    current_dir = Path.cwd().resolve()
    candidates = [
        current_dir / "data" / "Raw",
        current_dir / "data" / "raw",
        current_dir.parent / "data" / "Raw",
        current_dir.parent / "data" / "raw",
        current_dir.parent.parent / "data" / "Raw",
        current_dir.parent.parent / "data" / "raw",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "No PDF data folder found. Checked: " + ", ".join(str(p) for p in candidates)
    )


def load_documents():
    data_path = resolve_data_path()
    documents = []

    for pdf_path in sorted(data_path.glob("*.pdf")):
        reader = PdfReader(str(pdf_path))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        documents.append({"filename": pdf_path.name, "text": text})

    return documents

In [ ]:
if __name__ == "__main__":
    docs = load_documents()

    print(f"Loaded {len(docs)} documents")

    for doc in docs:
        print(f"{doc['filename']} -> {len(doc['text'])} characters")

In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])

In [ ]:
# Dependency is imported in the loader cell above.

In [ ]:
# Optional: run the loader from here if needed.

In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain"])

In [ ]:
docs = load_documents()

In [ ]:
print(f"Loaded {len(docs)} documents")

In [ ]:
for doc in docs:
    print(doc["filename"])

In [ ]:
print(docs[0]["text"][:1000])

In [ ]:
print(f"Loaded {len(docs)} documents")

In [ ]:
docs = load_documents()
print(f"Loaded {len(docs)} documents")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

chunks = []

for doc in docs:
    split_chunks = text_splitter.split_text(doc["text"])

    for idx, chunk in enumerate(split_chunks):
        chunks.append({
            "source": doc["filename"],
            "chunk_id": idx,
            "text": chunk
        })

print(f"Created {len(chunks)} chunks")

In [ ]:
print(chunks[0]["text"])

In [ ]:
print(chunks[1]["text"])

In [ ]:
print(chunks[0]["source"])
print(chunks[0]["chunk_id"])

In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])

In [ ]:
from openai import OpenAI
import os

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not set. Configure it in your environment or .env file.")

client = OpenAI(api_key=api_key)


In [ ]:
# Embeddings
if 'chunks' in globals() and chunks:
    texts = [chunk["text"] for chunk in chunks]

    embeddings = []

    for text in texts:
        try:
            response = client.embeddings.create(
                model="text-embedding-3-small",
                input=text
            )

            embeddings.append(response.data[0].embedding)
        except Exception as e:
            print("Embedding API error:", e)
            break

    print(f"Generated {len(embeddings)} embeddings")
    if embeddings:
        print(f"Embedding dimension: {len(embeddings[0])}")
else:
    print("No chunks available. Run the chunking cell first.")

In [ ]:
import pickle

with open("../data/processed/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Chunks saved successfully.")